# Kiểm tra model factory và training loop
Notebook đọc `configs/default.json` làm nguồn cấu hình chung. Các giá trị input/batch/workers của nhánh CIFAR-10 và training thật lấy từ cùng một bản cấu hình.

**Code đã được sửa nhưng chưa chạy lại; outputs cũ đã được sao lưu trước khi xóa để tránh nhầm bằng chứng.** Forward pretrained, smoke training và training đầy đủ đều có cờ riêng, mặc định tắt; chỉ cell kiểm tra batch CIFAR-10 được bật. Notebook 04 là nơi chuẩn bị/chạy thí nghiệm chính thức.

In [ ]:
from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone, timedelta
import hashlib
import os
import sys
import torch
import torchvision

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates
     if (path / "src/cifar_compare/train.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Không tìm thấy project.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cifar_compare.train import DEFAULTS, train_model
from src.cifar_compare.utils import load_config, resolve_device, set_seed

CONFIG_PATH = PROJECT_ROOT / "configs/default.json"
base_config = load_config(CONFIG_PATH, DEFAULTS)
config_fingerprint = hashlib.sha256(
    CONFIG_PATH.read_text(encoding="utf-8-sig").encode("utf-8")
).hexdigest()
DEVICE = resolve_device(base_config["device"])
INPUT_SIZE = int(base_config["input_size"])
NUM_CLASSES = int(base_config["num_classes"])
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)
print("torch / torchvision:", torch.__version__, "/", torchvision.__version__)
print("Device:", DEVICE)
print("Cấu hình chung từ JSON:", base_config)

## 1. Forward pretrained — tùy chọn
Vai trò: kiểm tra factory với kích thước/weights lấy từ JSON. Mặc định bỏ qua vì notebook 03 đã có bằng chứng; chỉ bật lại nếu cần kiểm tra phần bị thay đổi.

In [ ]:
RUN_FORWARD_CHECK = False

if RUN_FORWARD_CHECK:
    from src.cifar_compare.models import build_model, count_parameters
    for model_name in ("resnet18", "convnext_tiny"):
        model = build_model(
            model_name, num_classes=NUM_CLASSES,
            pretrained=bool(base_config["pretrained"]),
            weights_name=base_config["weights"],
            freeze_backbone=bool(base_config["freeze_backbone"]),
        ).to(DEVICE).eval()
        sample = torch.zeros(2, 3, INPUT_SIZE, INPUT_SIZE, device=DEVICE)
        with torch.inference_mode():
            logits = model(sample)
        assert tuple(logits.shape) == (2, NUM_CLASSES)
        assert torch.isfinite(logits).all().item()
        print(model_name, tuple(logits.shape), "parameters:", count_parameters(model))
        del model, sample, logits
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
else:
    print("SKIP forward: RUN_FORWARD_CHECK=False.")

## 2. Smoke training bằng tensor giả — tùy chọn
Vai trò: kiểm tra loop khi cần. Nhánh này sao chép cấu hình chung rồi **ghi đè có chủ đích** epochs=1, batch=2, workers=0, pretrained=False và output vào thư mục smoke. Không dùng các giá trị smoke cho nhánh CIFAR-10 hoặc kết quả chính thức.

In [ ]:
RUN_SMOKE_TRAINING = False
smoke_summary = None

if RUN_SMOKE_TRAINING:
    from torch.utils.data import DataLoader, TensorDataset
    smoke_config = deepcopy(base_config)
    smoke_config.update({
        "model": "resnet18", "pretrained": False,
        "epochs": 1, "batch_size": 2, "num_workers": 0,
        "early_stopping_patience": 0,
        "output_dir": "artifacts/notebook_smoke",
        "run_name": "resnet18_smoke_" + datetime.now(
            timezone(timedelta(hours=8))).strftime("%Y%m%d_%H%M%S_%f"),
    })
    set_seed(int(smoke_config["seed"]), bool(smoke_config["deterministic"]))
    images = torch.randn(4, 3, INPUT_SIZE, INPUT_SIZE)
    labels = torch.tensor([0, 1, 2, 3])
    fake_loader = DataLoader(TensorDataset(images, labels), batch_size=2)
    smoke_summary = train_model(
        "resnet18", smoke_config, fake_loader, fake_loader,
        ["airplane", "automobile", "bird", "cat", "deer",
         "dog", "frog", "horse", "ship", "truck"],
        device=resolve_device(smoke_config["device"]),
    )
    for field in ("best_checkpoint", "last_checkpoint", "metrics_csv"):
        assert Path(smoke_summary[field]).is_file()
    print("PASS smoke tensor giả:", smoke_summary)
else:
    print("SKIP smoke training: RUN_SMOKE_TRAINING=False.")

## 3. Đối chiếu cấu hình và batch CIFAR-10
Vai trò: dùng `data_config=deepcopy(base_config)`, không hard-code input/batch/workers. Chỉ đọc batch train; không đánh giá test. Nếu sửa JSON trong khi kernel đang mở, chạy lại từ cell đầu.

In [ ]:
RUN_CIFAR_DATA_CHECK = True
cifar_loaders = None
class_names = None
data_config = deepcopy(base_config)

if RUN_CIFAR_DATA_CHECK:
    from torchvision.datasets import CIFAR10
    from src.cifar_compare.data import get_dataloaders
    # Chặn tải ngoài ý muốn nếu dữ liệu local thiếu/hỏng.
    offline_check = CIFAR10(data_config["data_root"], train=True, download=False)
    del offline_check
    train_loader, val_loader, test_loader, class_names = get_dataloaders(data_config)
    cifar_loaders = (train_loader, val_loader, test_loader)
    batch_iterator = iter(train_loader)
    batch_images, batch_labels = next(batch_iterator)
    del batch_iterator
    print("Samples train/val/test:", len(train_loader.dataset),
          len(val_loader.dataset), len(test_loader.dataset))
    print("Classes:", class_names)
    print("input/batch/workers:", data_config["input_size"],
          data_config["batch_size"], data_config["num_workers"])
    print("Batch:", tuple(batch_images.shape), tuple(batch_labels.shape))
    assert batch_images.shape == (
        int(data_config["batch_size"]), 3,
        int(data_config["input_size"]), int(data_config["input_size"])
    )
    assert batch_images.dtype == torch.float32
    assert batch_labels.dtype == torch.int64
    assert torch.isfinite(batch_images).all().item()
    assert 0 <= batch_labels.min().item() <= batch_labels.max().item() < NUM_CLASSES
    print("PASS cấu hình DataLoader dùng cùng JSON.")
else:
    print("SKIP kiểm tra batch CIFAR-10.")

## 4. Training thật — mặc định tắt
Vai trò: nhánh cũ vẫn dùng cùng `data_config` của cell trên và tạo lại loaders để không dùng generator đã bị tiêu thụ khi kiểm tra batch. Khuyến nghị dùng notebook 04 cho run chính thức vì có preflight, báo cáo T015 và kiểm tra thư mục/dung lượng.

**Không bật nhánh này khi chỉ kiểm tra T013.**

In [ ]:
RUN_FULL_TRAINING = False
MODEL_TO_TRAIN = "resnet18"

if RUN_FULL_TRAINING:
    if cifar_loaders is None:
        raise RuntimeError("Chạy cell kiểm tra CIFAR-10 trước.")
    current_fingerprint = hashlib.sha256(
        CONFIG_PATH.read_text(encoding="utf-8-sig").encode("utf-8")
    ).hexdigest()
    if current_fingerprint != config_fingerprint:
        raise RuntimeError("JSON đã đổi: chạy lại notebook từ cell đầu.")
    final_config = deepcopy(data_config)
    final_config["model"] = MODEL_TO_TRAIN
    final_config["run_name"] = (
        MODEL_TO_TRAIN + "_seed" + str(final_config["seed"]) + "_"
        + datetime.now(timezone(timedelta(hours=8))).strftime("%Y%m%d_%H%M%S_%f")
    )
    set_seed(int(final_config["seed"]), bool(final_config["deterministic"]))
    fresh_train, fresh_val, fresh_test, fresh_classes = get_dataloaders(final_config)
    result = train_model(
        MODEL_TO_TRAIN, final_config, fresh_train, fresh_val, fresh_classes,
        device=resolve_device(final_config["device"]),
    )
    print(result)
else:
    print("SKIP training đầy đủ: RUN_FULL_TRAINING=False.")